# Live object detection on a camera feed

This notebook runs a YOLO model on a live camera, on a GPU, from a JupyterLab session you
started yourself on a compute node.

Everything it needs is staged at `/shared/workshops/basics`, so nothing downloads while you
run it. Work down the cells in order.

## 1. Check your GPU

You asked Slurm for a GPU when you started this session, so it is worth confirming you got one
before going further.

In [ ]:
import subprocess

import torch

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)

device = 0 if torch.cuda.is_available() else "cpu"

print("torch", torch.__version__)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

If `GPU available` comes back False, your job was allocated without one. Everything below still
runs, just slower.

## 2. Pick a camera

Three sources are available, and they refresh at very different rates, which matters once you get
to the live loop further down.

The New York City traffic cameras give you a new frame every couple of seconds. There are around
900 of them and they are the only source quick enough to actually look live, though at 352x240 they
are also the smallest.

The two Florida Tech cameras are 1920x1080 and much nicer pictures, but they only produce a new
frame about every eighty seconds. The Florida DOT cameras near campus are slower still. Those are
worth looking at, but they will not animate.

In [ ]:
import sys

sys.path.insert(0, "/shared/workshops/basics")

import feeds

for camera in feeds.list_cameras(limit=10):
    print(f"{camera['area']:<12} {camera['name']}")

The default above is the pedestrian walkway on the Brooklyn Bridge, which is busy with people and
refreshes every two seconds. To use a different one, search by name:

```python
camera = feeds.find("Crimson")
camera = feeds.CAMPUS
camera = feeds.find("Melbourne")
```

`feeds.find("Crimson")` looks over Babcock Street from the Crimson Crossing site and usually has
traffic in it. `feeds.CAMPUS` is the Olin Quad, the best picture of the three, though it looks down
on a mostly empty lawn from a long way up so there is often nothing in it to find. `"Melbourne"`
matches the Florida DOT cameras near campus, which print the road and mile marker into the frame.

To browse rather than search, use `feeds.list_cameras(area="Florida")`, or `area="Queens"` and the
other New York boroughs.

## 3. Grab a frame

Start with a single frame so you can see what the camera is pointed at.

In [ ]:
camera = feeds.find("Brooklyn Bridge - Ped")

frame = feeds.grab(camera)
print(camera["name"], frame.size)

frame

## 4. Run the model

YOLO takes an image and returns boxes with labels. The weights are already on disk, so this
loads from `/shared` rather than downloading.

One setting matters more than the rest here. YOLO shrinks every image to 640 pixels before
looking at it, and the campus cameras are 1920x1080, so a person forty metres from the lens ends
up a handful of pixels tall and the model either misses them or calls a palm tree a person.
Telling it to work at the camera's own resolution fixes that. The New York cameras are only
352x240, so they neither need nor benefit from it.

In [ ]:
from PIL import Image
from ultralytics import YOLO

model = YOLO("/shared/workshops/basics/models/yolov8s.pt")


def detect(frame, conf=0.35):
    imgsz = 1920 if max(frame.size) > 1000 else 640
    return model(frame, device=device, imgsz=imgsz, conf=conf, verbose=False)[0]


result = detect(frame)

Image.fromarray(result.plot()[:, :, ::-1])

Each box is one detection. To see what it actually found, count the labels.

In [ ]:
def count_labels(result):
    counts = {}
    for class_id in result.boxes.cls.tolist():
        label = result.names[int(class_id)]
        counts[label] = counts.get(label, 0) + 1
    return counts


count_labels(result)

## 5. Watch it live

This grabs a new frame every couple of seconds and re-runs the model, replacing the image each
time. Run the cell and watch, then interrupt the kernel with the stop button when you have seen
enough.

This only looks like video on the New York cameras. If you switch to a campus or Florida DOT
camera, raise `time.sleep(2)` to about `30`, because those produce a new frame roughly every eighty
seconds and the loop will otherwise redraw the same picture over and over.

In [ ]:
import time

from IPython.display import clear_output, display

for i in range(30):
    try:
        frame = feeds.grab(camera)
    except RuntimeError as error:
        print(f"frame {i + 1}: {error}")
        time.sleep(2)
        continue

    result = detect(frame)

    clear_output(wait=True)
    display(Image.fromarray(result.plot()[:, :, ::-1]))
    print(f"frame {i + 1} of 30    {count_labels(result)}")

    time.sleep(2)

## 6. Save a frame

Write the annotated frame to your home directory. In Section 4 of the tutorial you used `scp`
to copy files off the cluster, so this is a good chance to pull one back to your own machine.

In [ ]:
import os

annotated = Image.fromarray(result.plot()[:, :, ::-1])
annotated.save(os.path.expanduser("~/detection.png"))

print("saved to ~/detection.png")

## When you are done

Go back to the terminal where you submitted the job and cancel it with `scancel <jobid>`, or
close the session and let it hit its time limit. Either way the GPU goes back into the pool for
the next person.